In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter

print("Imports done!")

In [ ]:
MAX_FRAMES = 60
keypoints_dir = r"C:\Users\thiru\SETU\data\INCLUDE_landmarks\keypoints"

X_include = []
y_include = []

for category in os.listdir(keypoints_dir):
    category_path = os.path.join(keypoints_dir, category)
    for word in os.listdir(category_path):
        word_path = os.path.join(category_path, word)
        for parquet_file in os.listdir(word_path):
            if not parquet_file.endswith('.parquet'):
                continue
            file_path = os.path.join(word_path, parquet_file)
            df = pd.read_parquet(file_path)
            df = df[df['type'].isin(['pose', 'left_hand', 'right_hand'])]
            frames_keypoints = []
            for frame_num in sorted(df['frame'].unique()):
                frame_df = df[df['frame'] == frame_num]
                pose = frame_df[frame_df['type'] == 'pose'][['x','y','z']].values.flatten()
                if len(pose) < 99: pose = np.zeros(99)
                left = frame_df[frame_df['type'] == 'left_hand'][['x','y','z']].values.flatten()
                if len(left) < 63: left = np.zeros(63)
                right = frame_df[frame_df['type'] == 'right_hand'][['x','y','z']].values.flatten()
                if len(right) < 63: right = np.zeros(63)
                frames_keypoints.append(np.concatenate([pose, left, right]))
            kp = np.array(frames_keypoints)
            kp = np.nan_to_num(kp, nan=0.0)
            if len(kp) < MAX_FRAMES:
                pad = np.zeros((MAX_FRAMES - len(kp), 225))
                kp = np.vstack([kp, pad])
            else:
                kp = kp[:MAX_FRAMES]
            X_include.append(kp)
            y_include.append(word)

X_include = np.array(X_include)
y_include = np.array(y_include)

print(f"X shape: {X_include.shape}")
print(f"Total samples: {len(y_include)}")
print(f"Unique words: {len(np.unique(y_include))}")

In [ ]:
le_include = LabelEncoder()
y_encoded_include = le_include.fit_transform(y_include)
y_cat_include = to_categorical(y_encoded_include)

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_include, y_cat_include,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded_include
)

print(f"Classes: {len(le_include.classes_)}")
print(f"X_train: {X_train_i.shape}")
print(f"X_test:  {X_test_i.shape}")

In [ ]:
best_model = load_model(r"C:\Users\thiru\SETU\models\direction_a_include_best.keras")
print("Model loaded!")
best_model.summary()

In [ ]:
def predict_sign(parquet_path):
    df = pd.read_parquet(parquet_path)
    df = df[df['type'].isin(['pose', 'left_hand', 'right_hand'])]
    frames_keypoints = []
    for frame_num in sorted(df['frame'].unique()):
        frame_df = df[df['frame'] == frame_num]
        pose = frame_df[frame_df['type'] == 'pose'][['x','y','z']].values.flatten()
        if len(pose) < 99: pose = np.zeros(99)
        left = frame_df[frame_df['type'] == 'left_hand'][['x','y','z']].values.flatten()
        if len(left) < 63: left = np.zeros(63)
        right = frame_df[frame_df['type'] == 'right_hand'][['x','y','z']].values.flatten()
        if len(right) < 63: right = np.zeros(63)
        frames_keypoints.append(np.concatenate([pose, left, right]))
    kp = np.array(frames_keypoints)
    kp = np.nan_to_num(kp, nan=0.0)
    if len(kp) < MAX_FRAMES:
        pad = np.zeros((MAX_FRAMES - len(kp), 225))
        kp = np.vstack([kp, pad])
    else:
        kp = kp[:MAX_FRAMES]
    kp = kp.reshape(1, MAX_FRAMES, 225)
    prediction = best_model.predict(kp, verbose=0)
    predicted_index = np.argmax(prediction)
    confidence = prediction[0][predicted_index] * 100
    predicted_word = le_include.inverse_transform([predicted_index])[0]
    print(f"Predicted: {predicted_word} | Confidence: {confidence:.2f}%")
    return predicted_word, confidence

In [ ]:
import random

all_test_files = []
for category in os.listdir(keypoints_dir):
    category_path = os.path.join(keypoints_dir, category)
    for word in os.listdir(category_path):
        word_path = os.path.join(category_path, word)
        for f in os.listdir(word_path):
            if f.endswith('.parquet'):
                all_test_files.append((os.path.join(word_path, f), word))

random.seed(42)
test_samples = random.sample(all_test_files, 20)

correct = 0
for file_path, true_word in test_samples:
    predicted_word, confidence = predict_sign(file_path)
    is_correct = predicted_word == true_word
    if is_correct:
        correct += 1
    print(f"True: {true_word:20} | Predicted: {predicted_word:20} | Conf: {confidence:.1f}% | {'✅' if is_correct else '❌'}")

print(f"\nScore: {correct}/20 = {correct/20*100:.0f}%")